In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
from dask.distributed import Client, wait
from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
#import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import h5py
import copy

inFile = 'image_data/11092021_BaharehNP_3026CtlCorte.qptiff'
inFile_num = 3026
fabian_segmentation_file = '/home/users/browneld/lab/CODEX/codex-cns-devzone/figure-notebooks-playground/fabian_data/segmentation_files/data/11092021_BaharehNP_3026CtlCorte_IBA1_cell1.h5'


xstart = 100
xstop = 1200
ystart = 100
ystop = 1200

In [ ]:
#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors

# now applying special thresholding

In [ ]:

sample_stats = {}
sample_stats['inFile'] = inFile
sample_stats['inFile_num'] = inFile_num
(all_pages,shapes)=tiff_in(inFile)
markerDict = channelNames(inFile)
sample_stats['proteins'] = list(markerDict.values())
print(sample_stats['proteins'])

In [ ]:
LimitsPaulaorig = {'DAPI_1':0, 'Neurofilament':0, 'Vimentin':0, 'CD31':30, 'Claudin5':14, 'SMA':0, 'A-Beta':30, 'ApoE':60, 'Olig-2':30, 'NeuN':0, 'H2A.x':20, 'CollagenIV':30, 'PCNA':100, 'CD3e':70, 'CD4':80, 'MAP2':0, 'IBA1':16, 'TMEM119':0, 'CD163':60, 'HLA-DR':20, 'CD11c':0, 'CD68':50}

LimitsPaulacustom = {'DAPI_1':30, 'Neurofilament':10, 'Vimentin':20, 'Claudin5':24, 'SMA':25, 'A-Beta':30, 'ApoE':60, 'GFAP':30, 'Olig-2':30, 'PSD95':30, 'Synaptophysin':20, 'NeuN':25, 'H2A.x':25, 'CollagenIV':30, 'PCNA':100, 'CD3e':70, 'CD4':80, 'MAP2':20, 'IBA1':23, 'TMEM119':40, 'CD163':40, 'HLA-DR':20, 'CD11c':75, 'CD68':30}

limit_choice = LimitsPaulacustom

for gene_name in limit_choice.keys():
    if gene_name not in sample_stats['proteins']:
        print(gene_name + ' not found')


sample_stats['proteins_used_in_clustering'] = list(limit_choice.keys())
sample_stats['detection_mins_used_in_clustering'] = list(limit_choice.values())
convolve_data = False
downscale_reduction = 1
sample_stats['reduction'] = downscale_reduction
mD = limit_choice
gaussian_filter = False
d = {}
shapes = []
dbinar = {}
for keyID in markerDict.keys():
    print(keyID)
    key = markerDict[keyID]
    if key in mD.keys():
         
        orig_Img = all_pages[0][keyID]
        print(orig_Img.shape)
        plot1 = plt.hist(orig_Img.flatten(), bins=100)
        tMin = mD[key]
        
        if (gaussian_filter):
            print('applying gaussian filter')
            orig_Img = gaussian(orig_Img, sigma=2, preserve_range=True)
        
            
        if convolve_data:
            print('reducing_dims')
            orig_Img = transform.pyramid_reduce(orig_Img, downscale = downscale_reduction, preserve_range=True)
            
        
        flat = orig_Img.flatten()
        flat = flat[flat >= tMin]
        
        a_max = min(np.percentile(flat, q = 99), tMin + 30)
        plt.axvline(x=tMin, color = 'black')
        plt.axvline(x=a_max, color = 'black')
        plt.yscale('log')
        plt.show()
        
        plot2 = imshow(orig_Img == 0, origin = 'lower')
        plt.show()
        
        
        inImg = np.clip(a=orig_Img,a_min=0, a_max = a_max)
        inImg[inImg < tMin] = 0

        shapes.append(inImg.shape)
        
        
        inImg = (inImg - inImg.min())/(inImg.max() - inImg.min())
        
        inImg = inImg.round(decimals = 3)
        d[key] = inImg.flatten()
        plot3 = imshow(inImg == 0, origin = 'lower')
        plt.show()
        print("Done clipping and thresholding",markerDict[keyID])
        
        
        inImg_binar = np.clip(a=orig_Img,a_min=0, a_max = a_max)
        inImg_binar[inImg_binar < tMin] = 0
        inImg_binar[inImg_binar > 0] = 1
        dbinar[key] = inImg_binar.flatten()
        
        
        
        
        
        
        
        
        
        
             
shape = shapes[0]
sample_stats['shape'] = shape
d['X'] = np.indices(shape)[0].flatten()
d['Y'] = np.indices(shape)[1].flatten()

df_sub = pd.DataFrame(data=d)


dbinar['X'] = np.indices(shape)[0].flatten()
dbinar['Y'] = np.indices(shape)[1].flatten()

df_sub_binar = pd.DataFrame(data = dbinar)

In [ ]:
for name in limit_choice.keys():
    print(name)
    imgdata = np.asarray(df_sub[name]).reshape(sample_stats['shape'])
    gooby = plt.imshow(imgdata[xstart:xstop,ystart:ystop], origin = 'lower')
    plt.show()
    gooby = plt.imshow(imgdata, origin='lower')
    plt.show()
    
    


In [ ]:
df_file_path = 'pre_clustering_dfs/' + str(sample_stats['inFile_num']) + '_pre_clustering_df.csv'

df_sub.to_csv(df_file_path)  

dfbinar_file_path = 'pre_clustering_dfs/' + str(sample_stats['inFile_num']) + '_pre_clustering_dfbinar.csv'

df_sub_binar.to_csv(dfbinar_file_path)

savepath = 'saved_clustering_images_and_tables/'
with open(savepath + str(sample_stats['inFile_num']) + '_sample_stats.txt', 'w') as dict_file:
            dict_file.write(json.dumps(sample_stats))


# load data

In [ ]:

savepath = 'saved_clustering_images_and_tables/'
with open(savepath + str(inFile_num) + '_sample_stats.txt') as f:
    data = f.read()
sample_stats = json.loads(data)
        
df_file_path = 'pre_clustering_dfs/' + str(sample_stats['inFile_num']) + '_pre_clustering_df.csv'
df_sub = pd.read_csv(df_file_path, index_col=0)

dfbinar_file_path = 'pre_clustering_dfs/' + str(sample_stats['inFile_num']) + '_pre_clustering_dfbinar.csv'
dfbinar_sub = pd.read_csv(dfbinar_file_path, index_col=0)



In [ ]:
dfbinar_sub

# add custom channels

In [ ]:

dfbinar_sub['SMA - Claudin5 +'] = (dfbinar_sub['SMA'] == 0) & (dfbinar_sub['Claudin5'] == 1)
dfbinar_sub['GFAP + Vimentin +'] = (dfbinar_sub['GFAP'] == 1) & (dfbinar_sub['Vimentin'] == 1)


# add otsm annotations

In [ ]:
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict

In [ ]:
# read in fabian MG segmentations
with h5py.File(fabian_segmentation_file, 'r') as file:
    loaded_matrix = np.array(file['matrix'])
    
loaded_matrix = loaded_matrix.T
print(plt.imshow(loaded_matrix, origin = 'lower'))

dfbinar_sub['otsm_masks'] = loaded_matrix.flatten()

# read in segmentation metadata

fabian_data = anndata.read_h5ad('/home/users/browneld/lab/CODEX/codex-cns-devzone/figure-notebooks-playground/generate_fabian_seg_metadata/fabian_metadata.h5ad')
fabian_metadata = fabian_data.obs

category_to_color = {'Grey Matter': 'red', 'White Matter': 'blue', np.NaN: 'Gray'}



# build new column in df sub for mapping

dfbinar_sub['ImageID_CellID_man'] = str(sample_stats['inFile_num']) + '_' + (dfbinar_sub['otsm_masks'].astype(int)).astype(str)
meta_sub = fabian_metadata[['ImageID_CellID', 'Parent']].set_index('ImageID_CellID')


keys = fabian_metadata['ImageID_CellID']
values = meta_sub['Parent']
dictionary = dict(zip(keys, values))
dfbinar_sub['Parent_from_metadata'] = [dictionary.get(val, -1) for val in dfbinar_sub['ImageID_CellID_man'].tolist()]

viz_col = copy.deepcopy(dfbinar_sub.otsm_masks)
viz_col[viz_col > 0] = 1
dfbinar_sub['viz_col'] = viz_col

# collect centroid locations

In [ ]:

meta_only_img = fabian_metadata[fabian_metadata['ImageIDOLD'] == str(sample_stats['inFile_num'])]
centroid_x = meta_only_img['spatial_X']/sample_stats['reduction']
centroid_y = meta_only_img['spatial_Y']/sample_stats['reduction']

color_vector = [category_to_color[category] for category in meta_only_img.Parent]
print(plt.scatter(x = meta_only_img.spatial_X, y = meta_only_img.spatial_Y, c = color_vector, s = 0.5))


imageid_cellid = meta_only_img['ImageID_CellID']
parent = meta_only_img['Parent']

In [ ]:
imgdata = np.asarray(dfbinar_sub['viz_col']).reshape([sample_stats['shape'][0], sample_stats['shape'][1]])
implot = imshow(imgdata, origin = 'lower')
plt.show()

In [ ]:
imgdata = np.asarray(dfbinar_sub['viz_col']).reshape([sample_stats['shape'][0], sample_stats['shape'][1]])
implot = imshow(imgdata[xstart:xstop,ystart:ystop])

d = {'centroid_x': centroid_x, 'centroid_y': centroid_y}

neighborhood_table = pd.DataFrame(data=d)
#neighborhood_table = neighborhood_table[neighborhood_table['centroid_x'] < 1200]
neighborhood_table = neighborhood_table[(neighborhood_table['centroid_x'] < xstop) & (neighborhood_table['centroid_x'] > xstart)]

neighborhood_table = neighborhood_table[(neighborhood_table['centroid_y'] < ystop) & (neighborhood_table['centroid_y'] > ystart)]
plt.show()
plt.scatter(neighborhood_table['centroid_x'], neighborhood_table['centroid_y'])
plt.show()
implot = imshow(imgdata[xstart:xstop,ystart:ystop], origin = 'lower')
plt.scatter(neighborhood_table['centroid_x'] - xstart, neighborhood_table['centroid_y'] - xstart)
plt.show()


imgdata = np.asarray(dfbinar_sub['viz_col']).reshape([sample_stats['shape'][0], sample_stats['shape'][1]])
implot = imshow(imgdata, origin = 'lower')
plt.show()

# create cKDTrees from otsm points

In [ ]:
dfbinar_xy = dfbinar_sub[['Y', 'X']].to_numpy()

point_tree = scispatial.cKDTree(dfbinar_xy)
distance = 100
sample_stats['distance_used_in_neighborhood_analysis'] = distance

centers = np.column_stack((centroid_x, centroid_y))
neighbors = [point_tree.query_ball_point(centroid, distance) for centroid in centers]

d = {'imageid_cellid': imageid_cellid, 'parent':parent, 'centroid_x': centroid_x, 'centroid_y': centroid_y, 'neighbors': neighbors}
neighborhood_table = pd.DataFrame(data=d)

neighborhood_table['neighbor_pixel_ids'] = [dfbinar_sub.iloc[n].ImageID_CellID_man.tolist() for n in neighborhood_table.neighbors]


non_microglia_neighbors = []
non_microglia_neighbors_longboi = []

overlap_neighbors = []
overlap_neighbors_longboi = []

for ind in neighborhood_table.index:
    row = neighborhood_table.loc[ind]
    neighbors = row.neighbors
    neighbor_pixel_ids = row.neighbor_pixel_ids
    centroid_pixel_id = row.imageid_cellid
    neighbor_pixel_bools = [i != centroid_pixel_id for i in neighbor_pixel_ids]
    new_list = [i for (i,v) in zip (neighbors, neighbor_pixel_bools) if v]
    overlap_list = [i for (i,v) in zip (neighbors, neighbor_pixel_bools) if not v]
    non_microglia_neighbors.append(new_list)
    non_microglia_neighbors_longboi += new_list
    
    overlap_neighbors.append(overlap_list)
    overlap_neighbors_longboi += overlap_list

neighborhood_table['non_microglia_neighbors'] = non_microglia_neighbors
neighborhood_table['overlap_neighbors'] = overlap_neighbors





In [ ]:

dfbinar_sub['is_neighborhood'] = 0
dfbinar_sub.is_neighborhood.iloc[non_microglia_neighbors_longboi] = 1
imgdata = np.asarray(dfbinar_sub['is_neighborhood']).reshape(sample_stats['shape'])
implot = imshow(imgdata[xstart:xstop,ystart:ystop])

dfbinar_sub['is_neighborhood'] = 0
dfbinar_sub.is_neighborhood.iloc[overlap_neighbors_longboi] = 1
imgdata = np.asarray(dfbinar_sub['is_neighborhood']).reshape(sample_stats['shape'])
implot = imshow(imgdata[xstart:xstop,ystart:ystop])


# neighborhood analysis exlcluding mask

In [ ]:

#dfbinar_sub

columns_to_ignore = ['X', 'Y', 'otsm_masks', 'ImageID_CellID_man', 'otsm_leiden', 'is_neighborhood', 'Parent_from_metadata', 'viz_col']
columns_to_process = dfbinar_sub.columns[[i not in columns_to_ignore for i in dfbinar_sub.columns]]
neighbor_dict = {}
sample_stats['all_neighbor_names'] = columns_to_process

for col in columns_to_process:
    print(col)
    neighborhoods = [dfbinar_sub[col].get(i) for i in neighborhood_table.non_microglia_neighbors]
    neighborhood_percents = [sum(i)/len(i) for i in neighborhoods]
    neighbor_dict[col] = neighborhood_percents
    

neighbor_dict_all = {}

for col in columns_to_process:
    neighborhoods = [dfbinar_sub[col].get(i) for i in neighborhood_table.neighbors]
    neighborhood_percents = [sum(i)/len(i) for i in neighborhoods]
    neighbor_dict_all[col] = neighborhood_percents


neighbor_dict_direct_overlap = {}

for col in columns_to_process:
    neighborhoods = [dfbinar_sub[col].get(i) for i in neighborhood_table.overlap_neighbors]
    neighborhood_percents = [sum(i)/len(i) for i in neighborhoods]
    neighbor_dict_direct_overlap[col] = neighborhood_percents


    #neighbor_dict[col] = [dfbinar_sub[col].get(i) for i in neighborhood_table.non_microglia_neighbors]

neighbor_frame = pd.DataFrame.from_dict(neighbor_dict)
neighbor_adata = anndata.AnnData(neighbor_frame)
neighbor_adata.layers['X_all'] = pd.DataFrame.from_dict(neighbor_dict_all)
neighbor_adata.layers['X_direct_overlap'] = pd.DataFrame.from_dict(neighbor_dict_direct_overlap)

neighbor_adata.obs['id'] = list(neighborhood_table.imageid_cellid)
neighbor_adata.obs['X'] = list(neighborhood_table.centroid_x)
neighbor_adata.obs['Y'] = list(neighborhood_table.centroid_y)
neighbor_adata.obs['parent'] = list(neighborhood_table.parent)


neighbor_adata_GM = neighbor_adata[neighbor_adata.obs['parent'] == 'Grey Matter']
neighbor_adata_WM = neighbor_adata[neighbor_adata.obs['parent'] == 'White Matter']


ndf_GM = neighbor_adata_GM.to_df()
neighbor_adata_GM.layers['zscore'] = ndf_GM.apply(zscore).clip(lower=-2, upper = 2)
neighbor_adata_GM.layers['minmax'] = (ndf_GM-ndf_GM.min())/(ndf_GM.max()-ndf_GM.min())

ndf_GM_all = neighbor_adata_GM.to_df(layer = 'X_all')
neighbor_adata_GM.layers['zscore_all'] = ndf_GM_all.apply(zscore).clip(lower=-2, upper = 2)
neighbor_adata_GM.layers['minmax_all'] = (ndf_GM_all-ndf_GM_all.min())/(ndf_GM_all.max()-ndf_GM_all.min())

ndf_GM_direct = neighbor_adata_GM.to_df(layer = 'X_direct_overlap')
neighbor_adata_GM.layers['zscore_direct_overlap'] = ndf_GM_direct.apply(zscore).clip(lower=-2, upper = 2)
neighbor_adata_GM.layers['minmax_direct_overlap'] = (ndf_GM_direct-ndf_GM_direct.min())/(ndf_GM_direct.max()-ndf_GM_direct.min())

ndf_GM['parent'] = neighbor_adata_GM.obs['parent']


ndf_WM = neighbor_adata_WM.to_df()
neighbor_adata_WM.layers['zscore'] = ndf_WM.apply(zscore).clip(lower=-2, upper = 2)
neighbor_adata_WM.layers['minmax'] = (ndf_WM-ndf_WM.min())/(ndf_WM.max()-ndf_WM.min())

ndf_WM_all = neighbor_adata_WM.to_df(layer='X_all')
neighbor_adata_WM.layers['zscore_all'] = ndf_WM_all.apply(zscore).clip(lower=-2, upper=2)
neighbor_adata_WM.layers['minmax_all'] = (ndf_WM_all - ndf_WM_all.min()) / (ndf_WM_all.max() - ndf_WM_all.min())

ndf_WM_direct = neighbor_adata_WM.to_df(layer = 'X_direct_overlap')
neighbor_adata_WM.layers['zscore_direct_overlap'] = ndf_WM_direct.apply(zscore).clip(lower=-2, upper = 2)
neighbor_adata_WM.layers['minmax_direct_overlap'] = (ndf_WM_direct-ndf_WM_direct.min())/(ndf_WM_direct.max()-ndf_WM_direct.min())

ndf_WM['parent'] = neighbor_adata_WM.obs['parent']





neighbor_adata_WM.write_h5ad('coordinate_analysis/sample_adatas/' + str(sample_stats['inFile_num']) + "_WM.h5ad")
neighbor_adata_GM.write_h5ad('coordinate_analysis/sample_adatas/' + str(sample_stats['inFile_num']) + "_GM.h5ad")
#neighbor_frame_norm = (neighbor_frame-neighbor_frame.min())/(neighbor_frame.max()-neighbor_frame.min())



### heatmaps

In [ ]:

ndf_GMgrouped = ndf_GM.groupby('leiden').mean()
sns.clustermap(data=ndf_GMgrouped[sample_stats['all_neighbor_names']], figsize=(6,6))
sc.pl.heatmap(neighbor_adata_GM, groupby='leiden', var_names=neighbor_adata.var_names, layer='minmax')


ndf_WMgrouped = ndf_WM.groupby('leiden').mean()
sns.clustermap(data=ndf_WMgrouped[sample_stats['all_neighbor_names']], figsize=(6,6))
sc.pl.heatmap(neighbor_adata_WM, groupby='leiden', var_names=neighbor_adata.var_names, layer='minmax')




### box and whisker

In [ ]:
cMapDict = {'0': '#1f77b4',
 '1': '#ff7f0e',
 '2': '#279e68',
 '3': '#d62728',
 '4': '#aa40fc',
 '5': '#8c564b',
 '6': '#e377c2',
 '7': '#b5bd61',
 '8': '#17becf',
 '9': '#aec7e8',
 '10': '#ffbb78'}

for channel in sample_stats['all_neighbor_names']:
    sns.boxplot(data = ndf_GM, y = channel, x = 'leiden', order=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10"], showfliers=False, palette=cMapDict)
    plt.title(channel + ', sample ' + str(sample_stats['inFile_num']) + ', Grey Matter')

    plt.show()

for channel in sample_stats['all_neighbor_names']:
    sns.boxplot(data = ndf_WM, y = channel, x = 'leiden', order=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10"], showfliers=False, palette=cMapDict)
    plt.title(channel + ', sample ' + str(sample_stats['inFile_num']) + ', White Matter')

    plt.show()




# query other cell types

In [ ]:
fig6_data = pd.read_csv('../fig6_stardist_metadata.csv')

imageTypeMapQPv2 = {'2997': 'AD-2',
 '3155': 'AD-1',#Used for classification
 '3196': 'AD-3',
 '3280': 'AD-4',
 '1796': 'Ctl-3',
 '3628': 'Ctl-2',
 '3026': 'Ctl-1', #Used for classification
 '1873': 'Ctl-4'}

ImageID = imageTypeMapQPv2[str(sample_stats['inFile_num'])]


if (ImageID not in list(fig6_data['ImageID'])):
    print('image_cannot_be_found')


fig6_data_sub = fig6_data[(fig6_data['ImageID'] == ImageID)]





In [ ]:

centroid_x = fig6_data_sub['spatial_X']/sample_stats['reduction']
centroid_y = fig6_data_sub['spatial_Y']/sample_stats['reduction']
celltype = fig6_data_sub['phenotype']
parent = fig6_data_sub['Parent']


dfbinar_sub_xy = dfbinar_sub[['Y', 'X']].to_numpy()
point_tree = scispatial.cKDTree(dfbinar_sub_xy)
distance = 40

sample_stats['distance_used_in_neighborhood_analysis'] = distance

centers = np.column_stack((centroid_x, centroid_y))
neighbors = [point_tree.query_ball_point(centroid, distance) for centroid in centers]

d = {'celltype': celltype, 'parent':parent, 'centroid_x': centroid_x, 'centroid_y': centroid_y, 'neighbors': neighbors}
neighborhood_table = pd.DataFrame(data=d)

neighborhood_table['neighbor_pixel_ids'] = [dfbinar_sub.iloc[n].ImageID_CellID_man.tolist() for n in neighborhood_table.neighbors]

non_microglia_neighbors = []
non_microglia_neighbors_longboi = []

overlap_neighbors = []
overlap_neighbors_longboi = []

for ind in neighborhood_table.index:
    row = neighborhood_table.loc[ind]
    neighbors = row.neighbors
    neighbor_pixel_ids = row.neighbor_pixel_ids
    
    neighbor_pixel_bools = [i != centroid_pixel_id for i in neighbor_pixel_ids]
    new_list = [i for (i,v) in zip (neighbors, neighbor_pixel_bools) if v]
    overlap_list = [i for (i,v) in zip (neighbors, neighbor_pixel_bools) if not v]
    non_microglia_neighbors.append(new_list)
    non_microglia_neighbors_longboi += new_list
    
    overlap_neighbors.append(overlap_list)
    overlap_neighbors_longboi += overlap_list

neighborhood_table['non_microglia_neighbors'] = non_microglia_neighbors
neighborhood_table['overlap_neighbors'] = overlap_neighbors
columns_to_ignore = ['X', 'Y', 'otsm_masks', 'ImageID_CellID_man', 'otsm_leiden', 'is_neighborhood']
columns_to_process = dfbinar_sub.columns[[i not in columns_to_ignore for i in dfbinar_sub.columns]]
neighbor_dict = {}
sample_stats['all_neighbor_names'] = columns_to_process


for col in columns_to_process:
    print(col)
    neighborhoods = [dfbinar_sub[col].get(i) for i in neighborhood_table.non_microglia_neighbors]
    neighborhood_percents = [sum(i)/len(i) for i in neighborhoods]
    neighbor_dict[col] = neighborhood_percents
    

    #neighbor_dict[col] = [dfbinar_sub[col].get(i) for i in neighborhood_table.non_microglia_neighbors]

    
    
    
neighbor_frame = pd.DataFrame.from_dict(neighbor_dict)
neighbor_adata = anndata.AnnData(neighbor_frame)
neighbor_adata.obs['celltype'] = list(neighborhood_table.celltype)
neighbor_adata.obs['X'] = list(neighborhood_table.centroid_x)
neighbor_adata.obs['Y'] = list(neighborhood_table.centroid_y)
neighbor_adata.obs['parent'] = list(neighborhood_table.parent)


neighbor_adata_GM = neighbor_adata[neighbor_adata.obs['parent'] == 'Grey Matter']
neighbor_adata_WM = neighbor_adata[neighbor_adata.obs['parent'] == 'White Matter']


ndf_GM = neighbor_adata_GM.to_df()
neighbor_adata_GM.layers['zscore'] = ndf_GM.apply(zscore).clip(lower=-2, upper = 2)
neighbor_adata_GM.layers['minmax'] = (ndf_GM-ndf_GM.min())/(ndf_GM.max()-ndf_GM.min())
ndf_GM['celltype'] = neighbor_adata_GM.obs['celltype']
ndf_GM['parent'] = neighbor_adata_GM.obs['parent']


ndf_WM = neighbor_adata_WM.to_df()
neighbor_adata_WM.layers['zscore'] = ndf_WM.apply(zscore).clip(lower=-2, upper = 2)
neighbor_adata_WM.layers['minmax'] = (ndf_WM-ndf_WM.min())/(ndf_WM.max()-ndf_WM.min())
ndf_WM['celltype'] = neighbor_adata_WM.obs['celltype']
ndf_WM['parent'] = neighbor_adata_WM.obs['parent']







In [ ]:


for channel in sample_stats['all_neighbor_names']:
    sns.boxplot(data = ndf_GM, y = channel, x = 'celltype', showfliers=False)
    plt.title(channel + ', sample ' + str(sample_stats['inFile_num']) + ', Grey Matter')

    plt.show()
    
for channel in sample_stats['all_neighbor_names']:
    sns.boxplot(data = ndf_WM, y = channel, x = 'celltype', showfliers=False)
    plt.title(channel + ', sample ' + str(sample_stats['inFile_num']) + ', White Matter')

    plt.show()


In [ ]:
ImageID

In [ ]:
collections.Counter(fig6_data['ImageID'])

In [ ]:
fig6_data_sub = fig6_data[(fig6_data['ImageID'] == ImageID)]


In [ ]:
imageIDMapQP = {'10202021_BaharehNP_2997-ADCortex_EDTA.qptiff - resolution #1':'2997',
       '11162021_Bahareh-3155_EDTA.qptiff - resolution #1':'3155',
       '10122021_BaharehNP_ADCortex_EDTA.qptiff - resolution #1':'3196',
       '12032021-Bahareh_3280_Alz.qptiff - resolution #1':'3280',
       '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff - resolution #1':'1796',
       '12252021_Bahareh-3628-Cntr.qptiff - resolution #1':'3628',
       '11092021_BaharehNP_3026CtlCorte.qptiff - resolution #1':'3026',
       '10222021_BaharehNP_1873_CtrlCortex.qptiff - resolution #1':'1873'}
imageTypeMapQPv2 = {'2997': 'AD-2',
 '3155': 'AD-1',#Used for classification
 '3196': 'AD-3',
 '3280': 'AD-4',
 '1796': 'Ctl-3',
 '3628': 'Ctl-2',
 '3026': 'Ctl-1', #Used for classification
 '1873': 'Ctl-4'}
imageTypeMapQP = {'10202021_BaharehNP_2997-ADCortex_EDTA.qptiff - resolution #1':'AD',
       '11162021_Bahareh-3155_EDTA.qptiff - resolution #1':'AD',
       '10122021_BaharehNP_ADCortex_EDTA.qptiff - resolution #1':'AD',
       '12032021-Bahareh_3280_Alz.qptiff - resolution #1':'AD',
       '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff - resolution #1':'Ctl',
       '12252021_Bahareh-3628-Cntr.qptiff - resolution #1':'Ctl',
       '11092021_BaharehNP_3026CtlCorte.qptiff - resolution #1':'Ctl',
       '10222021_BaharehNP_1873_CtrlCortex.qptiff - resolution #1':'Ctl'}